In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pathlib import Path

# Cấu hình tối ưu cho Colab (12GB RAM)
spark = (SparkSession.builder
         .appName('Amazon_ETL_Pipeline')
         .master('local[*]')
         .config('spark.driver.memory', '8g')
         .config('spark.executor.memory', '4g')
         .config('spark.sql.execution.arrow.pyspark.enabled', 'true')
         .config('spark.sql.shuffle.partitions', '8')
         .getOrCreate())

# Tạo đường dẫn
RAW_DIR = Path('/content/amazon2023_raw/Electronics')
PARQUET_DIR = Path('/content/drive/MyDrive/amazon2023/Electronics')
RAW_DIR.mkdir(parents=True, exist_ok=True)
PARQUET_DIR.mkdir(parents=True, exist_ok=True)



In [ ]:
!wget -c https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/Electronics.jsonl.gz -O /content/amazon2023_raw/Electronics/reviews.jsonl.gz
!wget -c https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/meta_categories/meta_Electronics.jsonl.gz -O /content/amazon2023_raw/Electronics/meta.jsonl.gz

print("Tải dữ liệu hoàn tất.")

--2026-02-28 19:02:37--  https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/Electronics.jsonl.gz
Resolving mcauleylab.ucsd.edu (mcauleylab.ucsd.edu)... 137.110.161.5
Connecting to mcauleylab.ucsd.edu (mcauleylab.ucsd.edu)|137.110.161.5|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6474438619 (6.0G) [application/gzip]
Saving to: ‘/content/amazon2023_raw/Electronics/reviews.jsonl.gz’

/content/amazon2023 100%[===================>]   6.03G  47.7MB/s    in 2m 15s  

2026-02-28 19:04:52 (45.8 MB/s) - ‘/content/amazon2023_raw/Electronics/reviews.jsonl.gz’ saved [6474438619/6474438619]

--2026-02-28 19:04:53--  https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/meta_categories/meta_Electronics.jsonl.gz
Resolving mcauleylab.ucsd.edu (mcauleylab.ucsd.edu)... 137.110.161.5
Connecting to mcauleylab.ucsd.edu (mcauleylab.ucsd.edu)|137.110.161.5|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 13129004

In [4]:


from pyspark.sql.types import StructType, StructField, StringType, FloatType
from pyspark.sql.types import IntegerType, LongType, BooleanType, ArrayType
from pyspark.sql.window import Window

# Hàm chuẩn hoá NULL
def replace_missing_with_null(df):
    for col_name, dtype in df.dtypes:
        if dtype == 'string':
            df = df.withColumn(col_name,
                F.when(
                    F.col(col_name).isNull() | (F.trim(F.col(col_name)) == ''),
                    F.lit(None)
                ).otherwise(F.col(col_name)))
    return df

# ── Schema tường minh cho Reviews ───────────────────────────────────────────
REVIEWS_SCHEMA = StructType([
    StructField('user_id',           StringType(),  True),
    StructField('parent_asin',       StringType(),  True),
    StructField('rating',            FloatType(),   True),
    StructField('timestamp',         LongType(),    True),
    StructField('helpful_vote',      IntegerType(), True),
    StructField('verified_purchase', BooleanType(), True),
    StructField('title',             StringType(),  True),
    StructField('text',              StringType(),  True),
])

# ── Schema tường minh cho Metadata ──────────────────────────────────────────
# FIX: Khai báo rõ schema → Spark KHÔNG inferSchema
#      → KHÔNG đọc cột 'alert type' → KHÔNG bị lỗi COLUMN_ALREADY_EXISTS
META_SCHEMA = StructType([
    StructField('parent_asin',    StringType(),              True),
    StructField('title',          StringType(),              True),
    StructField('price',          FloatType(),               True),
])

print(' Đọc Reviews ...')
raw_reviews = (spark.read
               .schema(REVIEWS_SCHEMA)          # ← explicit schema
               .option('mode', 'DROPMALFORMED')
               .json(str(RAW_DIR / 'reviews.jsonl.gz')))

print(' Đọc Metadata ...')
raw_meta = (spark.read
            .schema(META_SCHEMA)               # ← explicit schema → fix lỗi
            .option('mode', 'DROPMALFORMED')
            .json(str(RAW_DIR / 'meta.jsonl.gz')))

# 2. Làm sạch NULL
reviews_df = replace_missing_with_null(raw_reviews)
meta_df    = replace_missing_with_null(raw_meta)

# 3 & 4. Thời gian & Kiểu dữ liệu
reviews_df = (reviews_df
    .withColumn('rating',       F.col('rating').cast(IntegerType()))
    .withColumn('timestamp_dt', F.to_timestamp(F.from_unixtime(F.col('timestamp') / 1000)))
    .withColumn('year',         F.year('timestamp_dt'))
    .withColumn('month',        F.month('timestamp_dt')))

# 5. Làm sạch text
reviews_df = (reviews_df
    .withColumn('review_text',
        F.trim(F.concat_ws(' ',
            F.coalesce(F.col('title'), F.lit('')),
            F.coalesce(F.col('text'),  F.lit('')))))
    .drop('title', 'text'))

# 6 & 7. Filter rác
reviews_df = reviews_df.filter(
    F.col('review_text').isNotNull() &
    (F.length(F.trim(F.col('review_text'))) > 0) &
    (F.col('verified_purchase') == True))

meta_df = meta_df.filter(
    F.col('price').isNotNull() &
    (F.col('price') > 0) &
    F.col('title').isNotNull() &
    (F.length(F.trim(F.col('title'))) > 0))

# 8. Cross-filter (Left Semi Join)
reviews_df = reviews_df.join(meta_df.select('parent_asin'), 'parent_asin', 'left_semi')
meta_df    = meta_df.join(reviews_df.select('parent_asin'), 'parent_asin', 'left_semi')

# 9. Xoá trùng & review_id
reviews_df = reviews_df.dropDuplicates(['user_id', 'parent_asin', 'timestamp'])
reviews_df = reviews_df.withColumn('review_id',
    F.row_number().over(Window.orderBy('timestamp')))
other_cols = [c for c in reviews_df.columns if c != 'review_id']
reviews_df = reviews_df.select('review_id', *other_cols)

# Ghi Parquet
print('\n Ghi Reviews Parquet ...')
(reviews_df
 .repartition(8, 'year', 'month')
 .write.mode('overwrite')
 .option('compression', 'snappy')
 .partitionBy('year', 'month')
 .parquet(str(PARQUET_DIR / 'reviews')))

print('Ghi Metadata Parquet ...')
(meta_df
 .repartition(4)
 .write.mode('overwrite')
 .option('compression', 'snappy')
 .parquet(str(PARQUET_DIR / 'metadata')))

print(f'\n Hoàn tất!')
print(f'   Reviews  : {reviews_df.count():,} rows')
print(f'   Metadata : {meta_df.count():,} rows')


 Đọc Reviews ...
 Đọc Metadata ...

 Ghi Reviews Parquet ...
Ghi Metadata Parquet ...

 Hoàn tất!
   Reviews  : 23,665,579 rows
   Metadata : 510,738 rows


In [ ]:
from huggingface_hub import HfApi, login

# Nhập thông tin của bạn
HF_WRITE_TOKEN = ""
USERNAME       = "tungne1311"

login(token=HF_WRITE_TOKEN)
api = HfApi()
REPO_ID = f"{USERNAME}/Amazon-Electronics-2023-Cleaned"

api.create_repo(repo_id=REPO_ID, repo_type="dataset", exist_ok=True)
api.upload_large_folder(folder_path=str(PARQUET_DIR / "reviews"), repo_id=REPO_ID, repo_type="dataset")
api.upload_large_folder(folder_path=str(PARQUET_DIR / "metadata"), repo_id=REPO_ID, repo_type="dataset")

Recovering from metadata files:   0%|          | 0/580 [00:00<?, ?it/s]




---------- 2026-02-28 21:07:55 (0:00:00) ----------
Files:   hashed 1/580 (8.0/4.5G) | pre-uploaded: 0/0 (0.0/4.5G) (+580 unsure) | committed: 0/580 (0.0/4.5G) | ignored: 0
Workers: hashing: 1 | get upload mode: 0 | pre-uploading: 0 | committing: 0 | waiting: 0
---------------------------------------------------


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...7d720.c000.snappy.parquet: 100%|##########| 46.9kB / 46.9kB            

...

  ...1-c000.snappy.parquet.crc: 100%|##########|  102kB /  102kB            

  ...c3551-c000.snappy.parquet:  61%|######1   | 7.99MB / 13.0MB            

  ...c3551-c000.snappy.parquet:  61%|######1   | 7.95MB / 13.0MB            

  ...c3551-c000.snappy.parquet:  61%|######1   | 7.98MB / 13.0MB            

  ...c3551-c000.snappy.parquet:  61%|######    | 7.95MB / 13.0MB            